 ### Agentic Pattern - Parallel Execution
 - Agent Personas for University Outreach
 - Scenario: Promoting a New AI Seminar Series
 - Agents can prepare email to the three differenct audience such as Faculty & Research, Students, and Industry people.

In [1]:
# import libraries
# Imports environment variables from a `.env` file.
from dotenv import load_dotenv

# Imports Agent, Runner, trace (for logging), and function_tool (for custom tools) from the agents module.
from agents import Agent, Runner, trace

# Imports type hint for streaming OpenAI responses.
from openai.types.responses import ResponseTextDeltaEvent

# Imports asyncio for running asynchronous tasks.
import asyncio

In [2]:

load_dotenv()

True

The backslash (\) in Python strings is used as a line continuation character. It tells Python that the string continues on the next line, allowing you to write long strings across multiple lines for better readability without breaking the string.

In [3]:
# Define instructions for university outreach agents
instructions1 = "You are an academic outreach coordinator in the Computer Science Department. \
You write formal and informative emails to faculty and researchers, inviting them to participate in or attend the department's AI and Ethics seminar series. \
Your emails emphasize the academic value, research relevance, and collaboration opportunities."

instructions2 = "You are a student engagement coordinator in the Computer Science Department. \
You write fun and motivating emails to students, encouraging them to attend the AI and Ethics seminar series. \
Your emails highlight exciting topics, career relevance, and opportunities to interact with experts."

instructions3 = "You are an industry liaison in the Computer Science Department. \
You write concise and professional emails to industry partners and alumni, inviting them to attend or support the AI and Ethics seminar series. \
Your emails focus on the practical impact of the research and opportunities for collaboration or sponsorship."


In [4]:
# Define the agents using the instructions
academic_outreach_agent = Agent(
    name="Academic Outreach Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

student_engagement_agent = Agent(
    name="Student Engagement Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

industry_liaison_agent = Agent(
    name="Industry Liaison Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)


### Runner.run_streamed()

- Use when you want to receive the response piece-by-piece as it's being generated.

- Useful for real-time display or longer replies.

#### Code Explanation:

This code initiates a streamed interaction with academic_outreach_agent, asking it to write an email. As the response is being generated in real-time, the code listens for each piece (or event) of the output. It filters only those events that contain actual text (raw_response_event with ResponseTextDeltaEvent) and prints each chunk immediately to the console using flush=True to ensure smooth, live output without delay. This creates a seamless streaming experience for viewing the email as it's written.

- ResponseTextDeltaEvent represents incremental chunks of text streamed by the model, allowing you to process or display the response as it is generated rather than waiting for the full output.

- end="" prevents line breaks when printing streamed text, allowing incremental output to appear as a continuous response.


- Synchronous: Tasks run one after another, and each task waits for the previous one to finish.


- Asynchronous: Tasks run without waiting, allowing multiple tasks to progress at the same time.

In [7]:
# Run the first agent using the Runner.run_streamed method
result = Runner.run_streamed(academic_outreach_agent, input="Write a professional email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Invitation to Participate in the AI and Ethics Seminar Series

Dear [Faculty/Researcher’s Name],

I hope this message finds you well. I am writing to invite you to participate in our upcoming AI and Ethics seminar series hosted by the Computer Science Department. This series aims to foster interdisciplinary dialogue and collaboration on the critical intersection of artificial intelligence and ethical considerations.

The seminars will feature a range of speakers from various fields, including computer science, philosophy, law, and social sciences, discussing the implications of AI technologies on society. We believe your expertise in [specific area of research or interest related to AI and ethics] would greatly enrich the discussions and inspire innovative approaches to these pressing issues.

Key details of the seminar series are as follows:

**Dates:** [Insert Dates]  
**Location:** [Insert Location/Virtual Platform Link]  
**Time:** [Insert Time]

Participation in this semi

In [8]:
print(result.final_output)

Subject: Invitation to Participate in our AI and Ethics Seminar Series

Dear [Recipient’s Name],

I hope this message finds you well. I am writing to invite you to participate in the upcoming AI and Ethics seminar series hosted by the Computer Science Department at [University Name]. This series aims to foster interdisciplinary dialogue around the ethical implications of artificial intelligence and its impact on society.

As a leading researcher in [specific area of research], your insights would greatly enrich our discussions and contribute to the collaborative atmosphere we aim to cultivate. The seminar series will feature a diverse lineup of speakers, including scholars, ethicists, and industry leaders, offering a platform for exploring current challenges and opportunities in AI ethics.

We believe your expertise aligns closely with the themes we will address, including fairness in algorithm design, bias mitigation, and the societal implications of automated decision-making. Partici

### Parallel Execution

 Code Explantion: This code runs three agents in parallel to generate emails using the same input message. The trace("Parallel emails") block is used to log or monitor the execution flow for better observability. asyncio.gather runs all three agents concurrently for faster response. Each agent processes the same message, and their final outputs are collected and printed separately with spacing for readability.

This code outputs = [result.final_output for result in results]

is equalant to

outputs = []

for result in results:

    outputs.append(result.final_output)
    




In [ ]:
# Running multiple agents in parallel
message = "Write a professional email"

# Run multiple agents in parallel while tracing the workflow execution
with trace("Parallel emails"):
    results = await asyncio.gather(
        Runner.run(academic_outreach_agent, message),  # Agent1
        Runner.run(student_engagement_agent, message), # Agent2
        Runner.run(industry_liaison_agent, message),   # Agent3  
    )
# Extract final text output from each agent result
outputs = [result.final_output for result in results]

# Print each generated email with spacing for readability
for output in outputs:
    print(output + "\n\n")

Subject: Invitation to Participate in Our AI and Ethics Seminar Series

Dear [Recipient's Name],

I hope this message finds you well. I am writing to extend an invitation to you and your team to participate in our upcoming seminar series focused on "AI and Ethics," hosted by the Computer Science Department.

As you know, the rapid advancements in artificial intelligence technologies pose profound ethical questions and challenges that necessitate thoughtful academic discourse and interdisciplinary collaboration. Our seminar series aims to bring together faculty, researchers, and students to explore these critical issues, share insights, and develop potential frameworks for responsible AI development and deployment.

The series will feature distinguished speakers from various disciplines, engaging discussions, and opportunities for participants to present their own research and viewpoints. We believe that your expertise in [specific area or topic related to the recipient’s work] would gr

In [ ]:
# Another way to process the results

# List of agent objects used to keep track of agent identity and execution order

agents = [
    academic_outreach_agent,
    student_engagement_agent,
    industry_liaison_agent,
]

# Run all agents in parallel on the same input message and collect their results
results = await asyncio.gather(
    Runner.run(academic_outreach_agent, message),
    Runner.run(student_engagement_agent, message),
    Runner.run(industry_liaison_agent, message),
)

# Pair each agent with its corresponding result and print the agent name and output
for agent, result in zip(agents, results):
    print(f"Agent: {agent.name}")
    print(result.final_output)
    print()


Agent: Academic Outreach Agent
Subject: Invitation to Participate in Our AI and Ethics Seminar Series

Dear [Recipient's Name],

I hope this email finds you well. I am writing to invite you to participate in our upcoming AI and Ethics seminar series, hosted by the Computer Science Department. This series aims to foster dialogue and collaboration among faculty and researchers who share an interest in the ethical implications and societal impacts of artificial intelligence.

As AI technologies continue to evolve and affect various aspects of our lives, it is crucial to engage in thoughtful discussions regarding their ethical frameworks. This seminar series will feature expert speakers from diverse fields, covering topics such as data bias, algorithmic accountability, and the implications of AI on privacy and security.

Here are the details of the seminar series:

- **Start Date:** [Insert Date]
- **Frequency:** [Insert Frequency, e.g., bi-weekly]
- **Location:** [Insert Location or Virtu